# Scenario sweep

Sweep `multiplier`. Plot per-party national seat counts.

In [ ]:
import os
from pathlib import Path

def _find_repo_root() -> Path:
    p = Path.cwd().resolve()
    for candidate in [p, *p.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("could not find repo root (no pyproject.toml in cwd or any parent)")

os.chdir(_find_repo_root())

def _latest_snapshot_hash() -> str:
    """Return the content hash of the most recent snapshot. Filenames are
    YYYY-MM-DD__v<schema>__<hash>.sqlite, so lexical sort = chronological."""
    snaps = sorted(Path("data/snapshots").glob("*.sqlite"))
    if not snaps:
        raise FileNotFoundError("no snapshots in data/snapshots/; run /snapshot first")
    return snaps[-1].stem.split("__")[-1]

def _pick_prediction(strategy_marker: str, label: str) -> Path:
    """Select the prediction file for the LATEST snapshot whose name contains
    strategy_marker AND label. Prediction filenames are
    <snap_hash>__<strategy>__<config_hash>__<label>.sqlite. Fails loud if 0 or
    >1 files match — the previous sorted(glob)[-1] silently picked an
    alphabetically-last file, which became nondeterministic once sweep_* runs
    or older snapshots' predictions shared the directory."""
    snap_hash = _latest_snapshot_hash()
    pred_dir = Path("data/predictions")
    # Match label as the trailing __{label}.sqlite suffix, not a substring.
    # Substring match collided when label='baseline_us' against files like
    # 'baseline_us_corrected' (introduced by --reform-polling-correction-pp runs).
    matches = [
        p for p in pred_dir.glob(f"{snap_hash}__*{strategy_marker}*.sqlite")
        if p.name.endswith(f"__{label}.sqlite")
    ]
    if not matches:
        raise FileNotFoundError(
            f"no prediction in {pred_dir} for snapshot {snap_hash} matching "
            f"strategy={strategy_marker!r} label={label!r}; run /predict first"
        )
    if len(matches) > 1:
        names = sorted(p.name for p in matches)
        raise RuntimeError(
            f"multiple predictions for snapshot {snap_hash} match "
            f"strategy={strategy_marker!r} label={label!r}: {names}; "
            f"remove duplicates or pass a more specific label"
        )
    return matches[0]

In [ ]:
from pathlib import Path
from prediction_engine.runner import run_prediction
from prediction_engine.analysis.sweep import collect_sweep
from schema.prediction import ReformThreatConfig

pred_dir = Path("data/predictions")
snap_path = sorted(Path("data/snapshots").glob("*.sqlite"))[-1]
sweep_paths = []
for m in (0.5, 0.75, 1.0, 1.25, 1.5):
    out = run_prediction(
        snapshot_path=snap_path,
        strategy_name="reform_threat_consolidation",
        scenario=ReformThreatConfig(multiplier=m),
        out_dir=pred_dir,
        label=f"sweep_m{m:.2f}".replace(".", "p"),
    )
    sweep_paths.append(out)

summary = collect_sweep(sweep_paths)
summary

In [ ]:
import matplotlib.pyplot as plt
pivot = summary.pivot(index="multiplier", columns="party", values="seats").fillna(0)
pivot.plot(figsize=(10, 5), marker="o")
plt.ylabel("Seats")
plt.title("National seat count vs reform-threat multiplier")
plt.show()

Reform's line should be monotonically decreasing; the consolidator parties' lines monotonically increasing.